In [1]:
from dotenv import load_dotenv
import os

from pymongo import MongoClient

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:

from preprocess_query.src import QueryPreprocessor

# Initialize
preprocessor = QueryPreprocessor(
    openai_api_key=OPENAI_API_KEY,
    openai_model="gpt-4o-mini",
    json_path="D:\LawAssistant\preprocess_query\dictionary.json"
)



<>:7: SyntaxWarning: invalid escape sequence '\L'
<>:7: SyntaxWarning: invalid escape sequence '\L'
C:\Users\ASUS\AppData\Local\Temp\ipykernel_6624\2086996032.py:7: SyntaxWarning: invalid escape sequence '\L'
  json_path="D:\LawAssistant\preprocess_query\dictionary.json"


Input:  Thủ tục đkkd & bhxh ko? Chi phí ntn?
Output: thủ tục đăng ký kinh doanh bao gồm việc nộp hồ sơ tại cơ quan đăng ký kinh doanh. bảo hiểm xã hội là nghĩa vụ của người sử dụng lao động. chi phí đăng ký kinh doanh và bảo hiểm xã hội phụ thuộc vào quy mô và loại hình doanh nghiệp.


In [6]:
from semantic_retrieval.src import SearchConfig, HybridSearchEngine
import logging

logging.basicConfig(level=logging.INFO)

config = SearchConfig(
    index_dir="D:/LawAssistant/semantic_retrieval/search_index",
    embedding_batch_size=256,
    processing_batch_size=500,
)

engine = HybridSearchEngine(config)
engine.load_index()


D:\uit_chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:semantic_retrieval.src.hybrid_search:Loading indexes...
INFO:semantic_retrieval.src.embedding_service:Loading embedding model: bkai-foundation-models/vietnamese-bi-encoder
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: bkai-foundation-models/vietnamese-bi-encoder
INFO:semantic_retrieval.src.embedding_service:Model loaded. Embedding dimension: 768
INFO:semantic_retrieval.src.faiss_index:FAISS index loaded: 21894 vectors
INFO:semantic_retrieval.src.bm25_index:BM25 index loaded: 21894 documents
INFO:semantic_retrieval.src.hybrid_search:Indexes loaded!


In [9]:
# Process query
query = "Xin hỏi LuatVietnam: Tôi có thửa đất do ông bà khai hoang để lại từ trước 1975, đến năm 1985 ông bà có cho người cháu có họ hàng gần để canh tác, sử dụng nhưng chỉ thỏa thuận miệng, và hàng năm bên mượn đất cũng không phải trả bất kỳ hoa màu gì cho ông bà tôi. Các giấy tờ chứng minh quyền sử dụng đất của ông bà đều không còn giữ. Đến năm 2022 gia đình người cháu kia đã tự đi xin cấp sổ đỏ cho mình và đã cấp được. Đến nay ông bà đi xin cấp sổ đỏ thì mới biết đất đã được cấp sổ cho người cháu kia. Vậy, luật sư cho tôi hỏi, trường hợp này ông bà tôi có đòi lại đất được không? Xin cảm ơn!"

results = engine.search(query, top_k=5)
print("search from query")
for result in results:
    print(result.metadata["full_path"])
    print("\n")

result = preprocessor.process(query)
print("preprocessed query")

print(f"Input:  {query}")
print()
print(f"Output: {result}")
print()
print("search from preprocessed query")

results = engine.search(result, top_k=5)

for result in results:
    print(result.metadata["full_path"])
    print("\n")

Batches: 100%|██████████| 1/1 [00:00<00:00,  9.09it/s]


search from query
52/2014/QH13_chương vi_điều 104_khoản 1


52/2014/QH13_chương vi_điều 104_khoản 2


52/2014/QH13_chương vii_điều 113_khoản 2


52/2014/QH13_chương vii_điều 113_khoản 1


101/2024/NĐ-CP_chương iii_mục 2_điều 26_khoản 2_điểm b




INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


preprocessed query
Input:  Xin hỏi LuatVietnam: Tôi có thửa đất do ông bà khai hoang để lại từ trước 1975, đến năm 1985 ông bà có cho người cháu có họ hàng gần để canh tác, sử dụng nhưng chỉ thỏa thuận miệng, và hàng năm bên mượn đất cũng không phải trả bất kỳ hoa màu gì cho ông bà tôi. Các giấy tờ chứng minh quyền sử dụng đất của ông bà đều không còn giữ. Đến năm 2022 gia đình người cháu kia đã tự đi xin cấp sổ đỏ cho mình và đã cấp được. Đến nay ông bà đi xin cấp sổ đỏ thì mới biết đất đã được cấp sổ cho người cháu kia. Vậy, luật sư cho tôi hỏi, trường hợp này ông bà tôi có đòi lại đất được không? Xin cảm ơn!

Output: Thửa đất do ông bà khai hoang để lại từ trước năm 1975. Năm 1985, ông bà cho người cháu canh tác, sử dụng theo thỏa thuận miệng. Người cháu không trả hoa màu cho ông bà. Giấy tờ chứng minh quyền sử dụng đất của ông bà không còn. Năm 2022, người cháu xin cấp sổ đỏ và được cấp. Ông bà đi xin cấp sổ đỏ thì phát hiện đất đã được cấp cho người cháu. Ông bà có quyền đòi lại đ

Batches: 100%|██████████| 1/1 [00:00<00:00,  9.22it/s]


52/2014/QH13_chương vi_điều 104_khoản 1


52/2014/QH13_chương vii_điều 113_khoản 1


52/2014/QH13_chương vi_điều 104_khoản 2


52/2014/QH13_chương vii_điều 113_khoản 2


52/2014/QH13_chương vii_điều 107_khoản 1


